# Student Performance Prediction

### Machine Learning Project — Linear Regression

**Objective:** Predict a student's final exam score using study hours, attendance, previous score, and sleep hours.

**Tech Stack:** Python · Pandas · NumPy · Matplotlib · Scikit-learn · Joblib

> **Dataset note:** This project uses a **synthetic dataset** generated for learning and demonstration. The dataset is deterministic (`random_state = 42`) and is not collected from real students.


## 1. Problem Statement

Student performance can be influenced by several measurable factors. In this project, we build a **Linear Regression** model to estimate a student's final score from four input features:

- `Hours_Studied`
- `Attendance`
- `Previous_Score`
- `Sleep_Hours`

The project follows a standard machine learning workflow:

**Data Generation → Data Inspection → EDA → Correlation Analysis → Train/Test Split → Model Training → Evaluation → Prediction → Model Saving**


## 2. Import Libraries

The libraries below are used for numerical computation, data analysis, visualization, machine learning, and model persistence.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

print("Libraries imported successfully.")

## 3. Generate the Synthetic Dataset

We create 500 student records with realistic ranges for the input features.

The target variable, `Final_Score`, is generated from a linear combination of the features with random noise. This makes the dataset suitable for demonstrating Linear Regression.

**Reproducibility:** `np.random.seed(42)` ensures the same dataset is generated each time.

In [ ]:
np.random.seed(42)

n = 500

hours_studied = np.random.randint(1, 11, n)
attendance = np.random.randint(50, 101, n)
previous_score = np.random.randint(40, 96, n)
sleep_hours = np.random.uniform(5, 9, n).round(1)

final_score = (
    3.5 * hours_studied
    + 0.25 * attendance
    + 0.45 * previous_score
    + 1.5 * sleep_hours
    + np.random.normal(0, 5, n)
)

final_score = np.clip(final_score, 0, 100).round(2)

df = pd.DataFrame({
    "Hours_Studied": hours_studied,
    "Attendance": attendance,
    "Previous_Score": previous_score,
    "Sleep_Hours": sleep_hours,
    "Final_Score": final_score
})

df.head()

In [ ]:
# Create project directories if they do not already exist.
data_dir = Path("data")
model_dir = Path("models")

data_dir.mkdir(exist_ok=True)
model_dir.mkdir(exist_ok=True)

dataset_path = data_dir / "student_performance.csv"
df.to_csv(dataset_path, index=False)

print(f"Dataset saved to: {dataset_path}")

## 4. Load and Inspect the Dataset

The saved CSV file is loaded again to simulate a real project workflow where data is read from an external file.

In [ ]:
df = pd.read_csv(dataset_path)

print(f"Dataset shape: {df.shape[0]} rows × {df.shape[1]} columns")
df.head()

In [ ]:
print("Dataset information:")
df.info()

In [ ]:
print("Missing values by column:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

In [ ]:
df.describe().round(2)

## 5. Exploratory Data Analysis (EDA)

EDA helps us understand the distribution of the data and identify relationships between the input features and the target variable.

### 5.1 Hours Studied vs Final Score

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df["Hours_Studied"], df["Final_Score"], alpha=0.7)

plt.xlabel("Hours Studied")
plt.ylabel("Final Score")
plt.title("Hours Studied vs Final Score")
plt.grid(alpha=0.2)
plt.show()

### 5.2 Attendance vs Final Score

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df["Attendance"], df["Final_Score"], alpha=0.7)

plt.xlabel("Attendance (%)")
plt.ylabel("Final Score")
plt.title("Attendance vs Final Score")
plt.grid(alpha=0.2)
plt.show()

### 5.3 Previous Score vs Final Score

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df["Previous_Score"], df["Final_Score"], alpha=0.7)

plt.xlabel("Previous Score")
plt.ylabel("Final Score")
plt.title("Previous Score vs Final Score")
plt.grid(alpha=0.2)
plt.show()

## 6. Correlation Analysis

Pearson correlation measures the strength and direction of a **linear relationship** between two numerical variables.

A value close to:

- `+1` → strong positive linear relationship
- `0` → little or no linear relationship
- `-1` → strong negative linear relationship

The target column correlates with itself at `1.0`, which is expected and should not be interpreted as a feature relationship.

In [ ]:
correlation = df.corr(numeric_only=True)

target_correlation = (
    correlation["Final_Score"]
    .drop("Final_Score")
    .sort_values(ascending=False)
)

print("Correlation with Final_Score:")
target_correlation.round(3)

In [ ]:
plt.figure(figsize=(8, 6))

plt.imshow(correlation, cmap="coolwarm", vmin=-1, vmax=1)
plt.colorbar(label="Correlation")

plt.xticks(range(len(correlation.columns)), correlation.columns, rotation=45, ha="right")
plt.yticks(range(len(correlation.columns)), correlation.columns)

plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

### EDA Insight

In this synthetic dataset, `Hours_Studied` has the strongest positive linear relationship with `Final_Score`, followed by `Previous_Score`, `Attendance`, and `Sleep_Hours`.

Correlation describes association; it does **not** by itself prove causation.

## 7. Feature Selection

### Features (X)
- `Hours_Studied`
- `Attendance`
- `Previous_Score`
- `Sleep_Hours`

### Target (y)
- `Final_Score`

In [ ]:
feature_columns = [
    "Hours_Studied",
    "Attendance",
    "Previous_Score",
    "Sleep_Hours"
]

X = df[feature_columns]
y = df["Final_Score"]

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

## 8. Train/Test Split

The dataset is divided into:

- **80% training data** → used to learn the model parameters
- **20% testing data** → used to evaluate the model on unseen data

`random_state=42` makes the split reproducible.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

## 9. Train the Linear Regression Model

Linear Regression learns a relationship between the input features and the continuous target variable.

The model is trained only on the training set.

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

print("Linear Regression model trained successfully.")

## 10. Generate Predictions

The trained model predicts final scores for the unseen test set.

In [ ]:
y_pred = model.predict(X_test)

comparison = pd.DataFrame({
    "Actual_Score": y_test.values,
    "Predicted_Score": y_pred.round(2)
})

comparison.head(10)

## 11. Model Evaluation

We use four regression metrics:

- **MAE (Mean Absolute Error):** average absolute prediction error.
- **MSE (Mean Squared Error):** squares errors, giving larger errors more weight.
- **RMSE (Root Mean Squared Error):** square root of MSE; it is in the same unit as the target.
- **R² Score:** proportion of target variance explained by the model.

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

metrics = pd.DataFrame({
    "Metric": ["MAE", "MSE", "RMSE", "R² Score"],
    "Value": [mae, mse, rmse, r2]
})

metrics.round(4)

### Evaluation Result

The model achieves:

- **MAE:** 4.29
- **MSE:** 27.13
- **RMSE:** 5.21
- **R²:** 0.85

Because the dataset is synthetic and the target was generated using a linear formula, these results are primarily useful for demonstrating the machine learning workflow. They should not be presented as real-world student-performance accuracy.

## 12. Actual vs Predicted Scores

Points closer to the ideal diagonal relationship indicate predictions that are closer to the actual values.

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(y_test, y_pred, alpha=0.7)

min_score = min(y_test.min(), y_pred.min())
max_score = max(y_test.max(), y_pred.max())
plt.plot([min_score, max_score], [min_score, max_score], linestyle="--")

plt.xlabel("Actual Final Score")
plt.ylabel("Predicted Final Score")
plt.title("Actual vs Predicted Final Scores")
plt.grid(alpha=0.2)
plt.show()

## 13. Residual Analysis

A residual is:

**Residual = Actual Score − Predicted Score**

A residual plot helps us inspect whether prediction errors are randomly distributed around zero.

In [ ]:
residuals = y_test - y_pred

plt.figure(figsize=(8, 5))
plt.scatter(y_pred, residuals, alpha=0.7)
plt.axhline(y=0, linestyle="--")

plt.xlabel("Predicted Final Score")
plt.ylabel("Residual")
plt.title("Residual Plot")
plt.grid(alpha=0.2)
plt.show()

## 14. Model Coefficients

Linear Regression assigns a coefficient to each feature. The coefficient represents the expected change in predicted `Final_Score` for a one-unit increase in that feature, **while keeping the other features constant**.

In [ ]:
coefficients = pd.DataFrame({
    "Feature": feature_columns,
    "Coefficient": model.coef_
}).sort_values("Coefficient", ascending=False)

coefficients.round(4)

In [ ]:
print("Intercept:", round(model.intercept_, 4))

## 15. Sample Prediction

Let's use the trained model to predict the final score for a hypothetical student.

In [ ]:
student = pd.DataFrame({
    "Hours_Studied": [7],
    "Attendance": [85],
    "Previous_Score": [72],
    "Sleep_Hours": [7]
})

predicted_score = model.predict(student)[0]

print(f"Predicted Final Score: {predicted_score:.2f}")

## 16. Save the Trained Model

Saving the model allows us to reuse it later without retraining it from scratch.

In [ ]:
model_path = model_dir / "student_performance_model.pkl"

joblib.dump(model, model_path)

print(f"Model saved to: {model_path}")

## 17. Load and Verify the Saved Model

We load the saved model and make the same prediction to verify that the model was persisted correctly.

In [ ]:
loaded_model = joblib.load(model_path)

loaded_prediction = loaded_model.predict(student)[0]

print(f"Prediction using saved model: {loaded_prediction:.2f}")
print("Prediction matches:", np.isclose(predicted_score, loaded_prediction))

## 18. Conclusion

This project demonstrates an end-to-end **supervised machine learning regression workflow**:

1. Generated and saved a synthetic dataset.
2. Inspected the dataset and checked data quality.
3. Performed exploratory data analysis and correlation analysis.
4. Selected relevant features and split the data into training and testing sets.
5. Trained a Linear Regression model.
6. Evaluated the model using MAE, MSE, RMSE, and R².
7. Visualized predictions and residuals.
8. Interpreted model coefficients.
9. Generated a sample prediction.
10. Saved and reloaded the trained model.

### Key Result

**R² Score: 0.85 | MAE: 4.29 | RMSE: 5.21**

> **Limitation:** The dataset is synthetic and designed for learning. Real-world performance would require a genuine, representative student dataset and appropriate validation.